# README
## Purpose
Execute an accelerated GSIB BERTopic train/validation/test pipeline variant.
## Inputs
- `Data/GSIB_model_input.csv`
## Outputs
- Faster experiment outputs with comparable split-aware BERTopic diagnostics.
## Notes
Designed to reduce runtime while preserving the chronological evaluation setup.

Checks Python, Torch, and CUDA/GPU environment availability.

In [1]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports core libraries for BERTopic modeling and preprocessing.

In [2]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loads the GSIB model input data and previews basic shape/head.

In [3]:
df = pd.read_csv('Data/GSIB_model_input.csv')
print('Loaded: Data/GSIB_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/GSIB_model_input.csv
Shape: (121186, 5)


,stock,date,date_only,headline,headline_raw
0,NDAQ,2014-02-17 05:11:00+00:00,2014-02-17,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...
1,NDAQ,2014-02-21 05:35:00+00:00,2014-02-21,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...
2,NDAQ,2014-04-23 12:39:00+00:00,2014-04-23,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
3,NDAQ,2014-05-15 10:23:00+00:00,2014-05-15,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
4,NDAQ,2014-05-26 05:59:00+00:00,2014-05-26,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...


Parses dates, deduplicates headlines, and creates chronological train/validation/test splits.

In [4]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define target split proportions (by document count, not date count)
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.15
SPLIT_TEST = 0.15

# 3. Calculate document-based boundaries (keep same day together for chronological integrity)
# Strategy: Find date boundaries such that document counts are close to target percentages
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)

# Find date indices where cumulative document count reaches target percentages
cumsum_docs = np.zeros(len(unique_dates) + 1)
for day_idx, unique_day in enumerate(unique_dates):
    n_docs_in_day = (day_code == day_idx).sum()
    cumsum_docs[day_idx + 1] = cumsum_docs[day_idx] + n_docs_in_day

total_docs = cumsum_docs[-1]
target_train_docs = total_docs * SPLIT_TRAIN
target_val_docs = total_docs * SPLIT_VAL

# Find date boundary indices
train_end_idx = np.searchsorted(cumsum_docs, target_train_docs, side='right') - 1
val_end_idx = np.searchsorted(cumsum_docs, target_train_docs + target_val_docs, side='right') - 1

# Ensure valid indices
train_end_idx = max(1, min(train_end_idx, len(unique_dates) - 2))
val_end_idx = max(train_end_idx + 1, min(val_end_idx, len(unique_dates) - 1))

# 4. Create masks based on date boundaries
mask_train = day_code < train_end_idx
mask_val = (day_code >= train_end_idx) & (day_code < val_end_idx)
mask_test = day_code >= val_end_idx

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)} ({100*len(train_docs)/after_dedup:.1f}%)")
print(f"Validation size: {len(val_docs)} ({100*len(val_docs)/after_dedup:.1f}%)")
print(f"Test size: {len(test_docs)} ({100*len(test_docs)/after_dedup:.1f}%)")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 121186 -> 90048 rows
Train size: 62969 (69.9%)
Validation size: 13503 (15.0%)
Test size: 13576 (15.1%)
Date ranges:
  Train: 2014-02-17 05:11:00+00:00 -> 2025-06-16 22:56:30+00:00
  Val:   2025-06-17 02:28:37+00:00 -> 2025-11-18 23:36:34+00:00
  Test:  2025-11-19 00:09:00+00:00 -> 2026-04-11 17:07:00+00:00


Configures embedding model, stopwords, and computes train embeddings.

In [5]:


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 919.33it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 984/984 [00:09<00:00, 98.77it/s] 



Embeddings computed on device: cuda


Builds and fits BERTopic on the training split.

In [6]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-04-12 21:54:28,171 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-04-12 21:54:56,664 - BERTopic - Dimensionality - Completed ✓
2026-04-12 21:54:56,666 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-12 21:54:56,666 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-12 22:38:30,982 - BERTopic - Cluster - Completed ✓
2026-04-12 22:38:31,003 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-12 22:38:30,982 - BERTopic - Cluster - Completed ✓
2026-04-12 22:38:31,003 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-12 22:38:31,742 - BERTopic - Representation - Completed ✓
2026-04-12 22:38:31,742 - BERTopic - Representation - Completed ✓



Model training complete!


Displays BERTopic topic summary table.

In [7]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,24828,-1_banking_says_street_bank,"[banking, says, street, bank, banks, ceo, wall...",[Indian Business Leaders Expect Major Profit a...
1,0,1393,0_ring_bell_opening_closing,"[ring, bell, opening, closing, celebration, na...",[Global X to Ring The Nasdaq Stock Market Clos...
2,1,836,1_blackstone_estate_real_reit,"[blackstone, estate, real, reit, breit, modwen...",[Blackstone to take Apartment Income REIT priv...
3,2,709,2_profit_beats_profits_estimates,"[profit, beats, profits, estimates, dealmaking...","[__COMPANY__ stock rises as profit, revenue be..."
4,3,693,3_dividend_yielding_aristocrats_yield,"[dividend, yielding, aristocrats, yield, tsx, ...","[3 High-Dividend Stocks to Buy in May, 3 High ..."
...,...,...,...,...,...
915,914,10,914_hurt_ib_aid_loans,"[hurt, ib, aid, loans, normalized, frc, weak, ...",[Loans &amp; Trading to Aid __COMPANY__ 's __T...
916,915,10,915_vulcan_underappreciated_doghouse_basket,"[vulcan, underappreciated, doghouse, basket, r...",[__COMPANY__ Stock: Undervalued and Underappre...
917,916,10,916_weinberger_dealmaker_poaches_dcm,"[weinberger, dealmaker, poaches, dcm, lavino, ...",[__COMPANY__ Hires Credit Suisse DCM Veterans ...
918,917,10,917_leadership_changes_servpro_responsibilities,"[leadership, changes, servpro, responsibilitie...",[__COMPANY__ Announces Senior Leadership Chang...


Projects validation documents into trained topic space.

In [8]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 422/422 [00:03<00:00, 123.69it/s]
2026-04-12 22:38:37,323 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.

2026-04-12 22:38:37,323 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-12 22:38:53,741 - BERTopic - Dimensionality - Completed ✓
2026-04-12 22:38:53,742 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-12 22:38:53,741 - BERTopic - Dimensionality - Completed ✓
2026-04-12 22:38:53,742 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-12 22:38:55,824 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-12 22:38:55,824 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-12 22:48:40,338 - BERTopic - Probabilities - Completed ✓
2026-04-12 22:48:40,339 - BERTopic - Cluster - Completed ✓
2026-04-12 22:48:40,338 - BERTopic - Probabilities - Completed ✓
2026-04-12 22:48:40,339 - 

Validation transform complete (test set remains untouched).


Defines evaluation utilities and computes coherence/diversity/similarity metrics.

In [9]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Numerical stability controls
COHERENCE_EPS = 1e-8
COHERENCE_MIN_TEXTS = 20
COHERENCE_MIN_VOCAB = 30
SIMILARITY_MAX_POINTS_PER_TOPIC = 300

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def safe_silhouette_score(
    embeddings,
    labels,
    outlier_label=-1,
    metric='cosine',
    min_cluster_size_for_metric=2,
    min_points_for_metric=10,
    drop_singletons=True
):
    if embeddings is None or labels is None:
        return np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < min_points_for_metric:
        return np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]

    if drop_singletons:
        unique_labels, counts = np.unique(labels_valid, return_counts=True)
        keep_labels = unique_labels[counts >= min_cluster_size_for_metric]
        if len(keep_labels) < 2:
            return np.nan
        keep_mask = np.isin(labels_valid, keep_labels)
        labels_valid = labels_valid[keep_mask]
        emb_valid = emb_valid[keep_mask]

    unique_labels, counts = np.unique(labels_valid, return_counts=True)
    if len(unique_labels) < 2:
        return np.nan
    if np.any(counts < min_cluster_size_for_metric):
        return np.nan
    if len(labels_valid) < max(min_points_for_metric, len(unique_labels) + 1):
        return np.nan

    try:
        value = float(silhouette_score(emb_valid, labels_valid, metric=metric))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def intra_inter_topic_similarity(
    embeddings,
    labels,
    outlier_label=-1,
    max_points_per_topic=SIMILARITY_MAX_POINTS_PER_TOPIC,
    random_state=42
):
    if embeddings is None or labels is None:
        return np.nan, np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan, np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan, np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < 2:
        return np.nan, np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]
    rng = np.random.default_rng(random_state)

    intra_values = []
    centroids = []

    for topic_label in np.unique(labels_valid):
        topic_mask = labels_valid == topic_label
        topic_emb = emb_valid[topic_mask]
        if topic_emb.shape[0] == 0:
            continue

        if topic_emb.shape[0] > max_points_per_topic:
            sampled_idx = rng.choice(topic_emb.shape[0], size=max_points_per_topic, replace=False)
            topic_emb = topic_emb[sampled_idx]

        centroids.append(topic_emb.mean(axis=0))

        if topic_emb.shape[0] >= 2:
            sim_matrix = cosine_similarity(topic_emb)
            upper_idx = np.triu_indices(sim_matrix.shape[0], k=1)
            if len(upper_idx[0]) > 0:
                intra_values.append(float(np.nanmean(sim_matrix[upper_idx])))

    intra_topic = float(np.nanmean(intra_values)) if len(intra_values) > 0 else np.nan

    if len(centroids) < 2:
        inter_topic = np.nan
    else:
        centroids_arr = np.vstack(centroids)
        centroid_sim = cosine_similarity(centroids_arr)
        upper_idx = np.triu_indices(centroid_sim.shape[0], k=1)
        inter_topic = float(np.nanmean(centroid_sim[upper_idx])) if len(upper_idx[0]) > 0 else np.nan

    return intra_topic, inter_topic

def coherence_score(topics, texts, dictionary, coherence_type='c_v', jitter=COHERENCE_EPS):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < COHERENCE_MIN_TEXTS or len(dictionary) < COHERENCE_MIN_VOCAB:
        return np.nan if jitter <= 0 else float(jitter)

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan if jitter <= 0 else float(jitter)

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan if jitter <= 0 else float(jitter)

    def _compute_value(kind):
        if kind == 'u_mass':
            corpus = [local_dict.doc2bow(doc) for doc in clean_texts]
            cm = CoherenceModel(
                topics=filtered_topics,
                corpus=corpus,
                dictionary=local_dict,
                coherence='u_mass'
            )
        else:
            cm = CoherenceModel(
                topics=filtered_topics,
                texts=clean_texts,
                dictionary=local_dict,
                coherence=kind
            )
        return float(cm.get_coherence())

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                value = _compute_value(coherence_type)
                if np.isfinite(value):
                    return float(value)
            except Exception:
                pass

            # Only c_v gets u_mass fallback. For c_npmi we avoid mixed-scale fallback.
            if coherence_type == 'c_v':
                try:
                    fallback_value = _compute_value('u_mass')
                    if np.isfinite(fallback_value):
                        return float(fallback_value)
                except Exception:
                    pass

    return np.nan if jitter <= 0 else float(jitter)

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
npmi_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_npmi')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
npmi_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_npmi')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

val_embeddings_for_metrics = sentence_model.encode(
    val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
sil_val = safe_silhouette_score(val_embeddings_for_metrics, val_topics)
sil_val = 0.0 if not np.isfinite(sil_val) else sil_val
intra_val, inter_val = intra_inter_topic_similarity(val_embeddings_for_metrics, val_topics)

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence NPMI (train): {npmi_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Coherence NPMI (val):   {npmi_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Validation silhouette (cosine, no outliers): {sil_val:.4f}")
print(f"Validation intra-topic similarity: {intra_val:.4f}")
print(f"Validation inter-topic similarity: {inter_val:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

Batches: 100%|██████████| 211/211 [00:02<00:00, 102.89it/s]



BERTopic Coherence C_v (train): 0.5168
BERTopic Coherence NPMI (train): 0.0897
BERTopic Coherence C_v (val):   0.3679
BERTopic Coherence NPMI (val):   -0.2131
BERTopic Topic Diversity (train): 0.6128
Validation outlier ratio (-1 topics): 0.5588
Validation silhouette (cosine, no outliers): 0.0785
Validation intra-topic similarity: 0.5513
Validation inter-topic similarity: 0.2945
Valid topic count for coherence: 919


## Tuning Strategy & Composite Scoring Explained

This cell runs a **comprehensive hyperparameter search** with a Jehnen-style multi-metric ranking setup.

### Search Design
- **Reduced Parameter Space**: 60 combinations instead of 900 (15x faster)
- **Folds**: 3 rolling/expanding time-series CV folds on train+val (test untouched).
- **Seed**: Fixed `RANDOM_SEED=42` for deterministic model selection.

### Per-Fold Metrics Used for Ranking (Equal Weighting)
| Metric | Direction | Notes |
|--------|-----------|-------|
| **c_v coherence** | Higher is better | Semantic coherence of topic words |
| **NPMI coherence** | Higher is better | Strong coherence signal for topic quality |
| **Topic diversity** | Higher is better | Fraction of unique words across topics |
| **Intra-topic similarity** | Higher is better | Cohesion within topic clusters in embedding space |
| **Inter-topic similarity** | Lower is better | Separation between topic clusters in embedding space |

### Aggregation & Normalization
1. Average each metric across folds per hyperparameter combination.
2. Min-max normalize each metric to [0,1].
3. Invert inter-topic similarity during normalization so lower raw inter-topic similarity gets higher normalized score.

### Composite Scoring Formula (No Manual Weighting)
All five ranking metrics are weighted equally:

$$
\text{score} = \frac{1}{5}\left(
\text{c\_v}_{\text{norm}} + \text{npmi}_{\text{norm}} + \text{diversity}_{\text{norm}} + \text{intra}_{\text{norm}} + \text{inter\_separation}_{\text{norm}}
\right)
$$

This is equivalent to **no preferential weighting** between ranking metrics.

### Model Selection
The highest composite score row is selected as `best_params`. Fold-level best rows are also reported to inspect temporal stability.

### Next: Multi-Seed Final Evaluation
After selecting best params, the final model is re-fit and evaluated across 9 random seeds on the untouched test set.

Runs rolling CV grid search and selects best BERTopic configuration.

In [10]:
from itertools import product
import time

# ================================================================================
# HYPERPARAMETER TUNING: GRID SEARCH + ROLLING CV + JEHNEN-STYLE EQUAL-WEIGHT RANKING
# ================================================================================
# Ranking metrics (equal weight):
#   - c_v coherence
#   - c_npmi coherence
#   - topic diversity
#   - intra-topic similarity (higher better)
#   - inter-topic similarity (lower better; inverted during normalization)
# ================================================================================

# -------------------- Time-series CV controls (fixed seed for model selection) --------------------
N_FOLDS = 3  # rolling/expanding buckets before test
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value, fallback=np.nan):
    try:
        v = float(value)
    except Exception:
        return fallback
    return v if np.isfinite(v) else fallback

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

param_grid = {
    'n_neighbors': [15, 30, 50],           # Reduced from [10, 30, 50, 70, 90]
    'n_components': [5, 10],                # Reduced from [4, 6, 8]
    'min_cluster_size': [10, 15, 20],       # Reduced from [8, 12, 16, 24, 32]
    'min_samples': [1, 3],                  # Reduced from [1, 2, 4, 6]
    'ngram_range': [(1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))
expected_combinations = (
    len(param_grid['n_neighbors'])
    * len(param_grid['n_components'])
    * len(param_grid['min_cluster_size'])
    * len(param_grid['min_samples'])
    * len(param_grid['ngram_range'])
)

print(f'Configured combinations: {expected_combinations} (target ~300)')
print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds (fixed seed={RANDOM_SEED})')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        start = time.perf_counter()

        umap_model_i = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            metric='cosine',
            random_state=RANDOM_SEED
        )
        hdbscan_model_i = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1
        )
        vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

        topic_model_i = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model_i,
            hdbscan_model=hdbscan_model_i,
            vectorizer_model=vectorizer_model_i,
            calculate_probabilities=False,
            verbose=False
        )

        _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
        val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

        topic_words_i = [
            [word for word, _ in topic_model_i.get_topic(t)]
            for t in topic_model_i.get_topics().keys()
            if t != -1 and topic_model_i.get_topic(t) is not None
        ]

        has_enough_for_coherence = (
            len(processed_val_fold) >= 20 and
            len(id2word_fold) >= 30 and
            len(topic_words_i) >= 2
        )

        cv_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        npmi_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_npmi', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        div_i = sanitize_metric(topic_diversity(topic_words_i), fallback=0.0)
        intra_i, inter_i = intra_inter_topic_similarity(fold_val_embeddings, val_topics_i)
        intra_i = sanitize_metric(intra_i, fallback=0.0)
        inter_i = sanitize_metric(inter_i, fallback=1.0)

        n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))
        elapsed = time.perf_counter() - start

        cv_rows.append({
            'trial': trial,
            'fold': fold['fold'],
            'random_seed': RANDOM_SEED,
            'n_neighbors': n_neighbors,
            'n_components': n_components,
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'ngram_range': str(ngram_range),
            'n_topics': n_topics_i,
            'cv_val': cv_val_i,
            'npmi_val': npmi_val_i,
            'topic_diversity': div_i,
            'intra_topic_similarity': intra_i,
            'inter_topic_similarity': inter_i,
            'fit_seconds': elapsed
        })

cv_results = pd.DataFrame(cv_rows)

agg_cols = [
    'cv_val', 'npmi_val', 'topic_diversity',
    'intra_topic_similarity', 'inter_topic_similarity',
    'n_topics', 'fit_seconds'
]
tuning_results = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[agg_cols].mean()

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['npmi_val_norm'] = minmax_norm(tuning_results['npmi_val'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['intra_topic_similarity_norm'] = minmax_norm(tuning_results['intra_topic_similarity'], higher_is_better=True)
tuning_results['inter_topic_separation_norm'] = minmax_norm(tuning_results['inter_topic_similarity'], higher_is_better=False)

# Equal weighting across Jehnen-style ranking metrics (no preferential weighting)
tuning_results['composite_score'] = (
    tuning_results['cv_val_norm'] +
    tuning_results['npmi_val_norm'] +
    tuning_results['topic_diversity_norm'] +
    tuning_results['intra_topic_similarity_norm'] +
    tuning_results['inter_topic_separation_norm']
) / 5.0

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['npmi_val_norm'] = fold_agg.groupby('fold')['npmi_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['intra_topic_similarity_norm'] = fold_agg.groupby('fold')['intra_topic_similarity'].transform(lambda x: minmax_norm(x, True))
fold_agg['inter_topic_separation_norm'] = fold_agg.groupby('fold')['inter_topic_similarity'].transform(lambda x: minmax_norm(x, False))

fold_agg['fold_score'] = (
    fold_agg['cv_val_norm'] +
    fold_agg['npmi_val_norm'] +
    fold_agg['topic_diversity_norm'] +
    fold_agg['intra_topic_similarity_norm'] +
    fold_agg['inter_topic_separation_norm']
) / 5.0

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
)

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV (fixed-seed model selection):')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 1195/1195 [00:11<00:00, 106.27it/s]



Generated 3 rolling folds (train+val pool only).
Fold 1: train 2014-02-17 00:00:00+00:00 -> 2018-06-15 00:00:00+00:00 | val 2018-06-19 00:00:00+00:00 -> 2021-07-20 00:00:00+00:00
Fold 2: train 2014-02-17 00:00:00+00:00 -> 2021-07-20 00:00:00+00:00 | val 2021-07-22 00:00:00+00:00 -> 2023-09-19 00:00:00+00:00
Fold 3: train 2014-02-17 00:00:00+00:00 -> 2023-09-19 00:00:00+00:00 | val 2023-09-21 00:00:00+00:00 -> 2025-11-18 00:00:00+00:00
Configured combinations: 36 (target ~300)
Running 36 parameter combinations x 3 folds (fixed seed=42)
[Trial 1/36] n_neighbors=15, n_components=5, min_cluster_size=10, min_samples=1, ngram_range=(1, 2)
[Trial 2/36] n_neighbors=15, n_components=5, min_cluster_size=10, min_samples=3, ngram_range=(1, 2)
[Trial 2/36] n_neighbors=15, n_components=5, min_cluster_size=10, min_samples=3, ngram_range=(1, 2)
[Trial 3/36] n_neighbors=15, n_components=5, min_cluster_size=15, min_samples=1, ngram_range=(1, 2)
[Trial 3/36] n_neighbors=15, n_components=5, min_cluster_si

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,30,10,20,3,"(1, 2)",0.575206,0.071558,0.844186,0.452469,0.417889,171.000000,72.472637,1.000000,1.000000,0.836904,0.356842,0.679781,0.774706
1,30,10,15,3,"(1, 2)",0.561163,0.063875,0.847276,0.466160,0.397624,222.666667,75.771036,0.658574,0.797372,0.881002,0.524925,0.456426,0.663659
2,50,10,10,3,"(1, 2)",0.564476,0.070030,0.824655,0.504857,0.356215,286.000000,84.249103,0.739114,0.959706,0.558187,1.000000,0.000000,0.651402
3,50,5,20,1,"(1, 2)",0.557140,0.054636,0.855615,0.456502,0.416177,172.666667,82.343141,0.560744,0.553700,1.000000,0.406350,0.660911,0.636341
4,50,5,20,3,"(1, 2)",0.558814,0.055172,0.854388,0.456734,0.406793,156.666667,80.721031,0.601452,0.567832,0.982490,0.409201,0.557483,0.623692
5,50,10,15,3,"(1, 2)",0.551522,0.054978,0.849036,0.493815,0.384408,196.333333,82.630227,0.424165,0.562726,0.906106,0.864435,0.310754,0.613637
6,50,5,15,3,"(1, 2)",0.555254,0.056265,0.849986,0.478221,0.385512,200.666667,86.363789,0.514906,0.596677,0.919665,0.672989,0.322919,0.605431
7,50,10,20,3,"(1, 2)",0.556052,0.049709,0.850103,0.460893,0.410523,153.666667,81.453276,0.534288,0.423779,0.921331,0.460263,0.598594,0.587651
8,50,10,20,1,"(1, 2)",0.553328,0.054705,0.840762,0.466814,0.401349,172.000000,81.802983,0.468059,0.555529,0.788034,0.532952,0.497483,0.568411
9,50,10,15,1,"(1, 2)",0.554893,0.046025,0.844940,0.487799,0.379141,226.000000,83.040257,0.506106,0.326618,0.847656,0.790582,0.252695,0.544731



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,fold_score
0,1,30,10,20,3,"(1, 2)",0.695306,0.168855,0.862963,0.374695,0.488067,0.760426
1,2,30,10,20,1,"(1, 2)",0.549210,0.028523,0.844628,0.454330,0.409163,0.739696
2,3,50,10,10,3,"(1, 2)",0.503980,0.042712,0.825920,0.553654,0.332855,0.783837



Average fit time per model run (seconds): 75.0
Best overall configuration selected from rolling CV (fixed-seed model selection):


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,30,10,20,3,"(1, 2)",0.575206,0.071558,0.844186,0.452469,0.417889,171.0,72.472637,1.0,1.0,0.836904,0.356842,0.679781,0.774706


Refits best config and evaluates seed-wise variability on test split.

In [11]:
from ast import literal_eval

# Refit best hyperparameter configuration on train+val, then evaluate variability across random seeds on test
EVAL_SEEDS = [42, 7, 123, 2024, 99, 11, 77, 314, 2718]

train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(
    train_val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
test_embeddings = sentence_model.encode(
    test_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

best_ngram = literal_eval(best_params['ngram_range'])
seed_rows = []

for seed in EVAL_SEEDS:
    print(f'Running final fit/eval for seed={seed}...')

    best_umap = UMAP(
        n_neighbors=int(best_params['n_neighbors']),
        n_components=int(best_params['n_components']),
        metric='cosine',
        random_state=seed
    )
    best_hdbscan = HDBSCAN(
        min_cluster_size=int(best_params['min_cluster_size']),
        min_samples=int(best_params['min_samples']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

    best_topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=best_umap,
        hdbscan_model=best_hdbscan,
        vectorizer_model=best_vectorizer,
        calculate_probabilities=True,
        verbose=False
    )

    _topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
    test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

    best_topic_words = [
        [word for word, _ in best_topic_model.get_topic(t)]
        for t in best_topic_model.get_topics().keys()
        if t != -1 and best_topic_model.get_topic(t) is not None
    ]

    cv_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    npmi_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_npmi', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    div_test = sanitize_metric(topic_diversity(best_topic_words), fallback=0.0)
    intra_test, inter_test = intra_inter_topic_similarity(test_embeddings, test_topics)
    intra_test = sanitize_metric(intra_test, fallback=0.0)
    inter_test = sanitize_metric(inter_test, fallback=1.0)

    # Final-only diagnostics (not used in tuning)
    sil_test = sanitize_metric(
        safe_silhouette_score(test_embeddings, test_topics, outlier_label=-1, metric='cosine'),
        fallback=0.0
    )
    test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
    test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

    seed_rows.append({
        'seed': seed,
        'cv_test': cv_test,
        'npmi_test': npmi_test,
        'topic_diversity_test': div_test,
        'intra_topic_similarity_test': intra_test,
        'inter_topic_similarity_test': inter_test,
        'test_silhouette': sil_test,
        'test_outlier_ratio': test_outlier_ratio,
        'test_topic_count': test_topic_count
    })

seed_results = pd.DataFrame(seed_rows)

print('\nPer-seed test metrics:')
display(seed_results)

summary_stats = pd.DataFrame([{
    'cv_test_mean': seed_results['cv_test'].mean(skipna=True),
    'cv_test_std': seed_results['cv_test'].std(skipna=True),
    'npmi_test_mean': seed_results['npmi_test'].mean(skipna=True),
    'npmi_test_std': seed_results['npmi_test'].std(skipna=True),
    'topic_diversity_mean': seed_results['topic_diversity_test'].mean(skipna=True),
    'topic_diversity_std': seed_results['topic_diversity_test'].std(skipna=True),
    'intra_topic_similarity_mean': seed_results['intra_topic_similarity_test'].mean(skipna=True),
    'intra_topic_similarity_std': seed_results['intra_topic_similarity_test'].std(skipna=True),
    'inter_topic_similarity_mean': seed_results['inter_topic_similarity_test'].mean(skipna=True),
    'inter_topic_similarity_std': seed_results['inter_topic_similarity_test'].std(skipna=True),
    'test_silhouette_mean': seed_results['test_silhouette'].mean(skipna=True),
    'test_silhouette_std': seed_results['test_silhouette'].std(skipna=True),
    'test_outlier_ratio_mean': seed_results['test_outlier_ratio'].mean(skipna=True),
    'test_outlier_ratio_std': seed_results['test_outlier_ratio'].std(skipna=True),
    'test_topic_count_mean': seed_results['test_topic_count'].mean(skipna=True),
    'test_topic_count_std': seed_results['test_topic_count'].std(skipna=True)
}])

print('\nFinal test metrics variability across random seeds:')
display(summary_stats)

def format_mean_std(mean_value, std_value, digits=4):
    if not np.isfinite(mean_value):
        return 'NA'
    if not np.isfinite(std_value):
        return f"{mean_value:.{digits}f} ± NA"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"

thesis_report = pd.DataFrame([{
    'Model': 'BERTopic (best CV config)',
    'Seeds (n)': len(EVAL_SEEDS),
    'Coherence C_v (test)': format_mean_std(summary_stats.at[0, 'cv_test_mean'], summary_stats.at[0, 'cv_test_std'], digits=4),
    'Coherence NPMI (test)': format_mean_std(summary_stats.at[0, 'npmi_test_mean'], summary_stats.at[0, 'npmi_test_std'], digits=4),
    'Topic Diversity (test)': format_mean_std(summary_stats.at[0, 'topic_diversity_mean'], summary_stats.at[0, 'topic_diversity_std'], digits=4),
    'Intra-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'intra_topic_similarity_mean'], summary_stats.at[0, 'intra_topic_similarity_std'], digits=4),
    'Inter-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'inter_topic_similarity_mean'], summary_stats.at[0, 'inter_topic_similarity_std'], digits=4),
    'Silhouette (test, cosine)': format_mean_std(summary_stats.at[0, 'test_silhouette_mean'], summary_stats.at[0, 'test_silhouette_std'], digits=4),
    'Outlier Ratio (test)': format_mean_std(summary_stats.at[0, 'test_outlier_ratio_mean'], summary_stats.at[0, 'test_outlier_ratio_std'], digits=4),
    'Topic Count (test)': format_mean_std(summary_stats.at[0, 'test_topic_count_mean'], summary_stats.at[0, 'test_topic_count_std'], digits=2)
}])

print('\nThesis-ready reporting table (mean ± std across seeds):')
display(thesis_report)

Batches: 100%|██████████| 1195/1195 [00:10<00:00, 114.14it/s]

Batches: 100%|██████████| 213/213 [00:02<00:00, 94.06it/s]



Running final fit/eval for seed=42...
Running final fit/eval for seed=7...
Running final fit/eval for seed=7...
Running final fit/eval for seed=123...
Running final fit/eval for seed=123...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=99...
Running final fit/eval for seed=99...
Running final fit/eval for seed=11...
Running final fit/eval for seed=11...
Running final fit/eval for seed=77...
Running final fit/eval for seed=77...
Running final fit/eval for seed=314...
Running final fit/eval for seed=314...
Running final fit/eval for seed=2718...
Running final fit/eval for seed=2718...

Per-seed test metrics:

Per-seed test metrics:


,seed,cv_test,npmi_test,topic_diversity_test,intra_topic_similarity_test,inter_topic_similarity_test,test_silhouette,test_outlier_ratio,test_topic_count
0,42,0.492526,-0.001989,0.807827,0.564155,0.328629,0.076342,0.583014,626
1,7,0.492971,0.000641,0.808847,0.550201,0.326557,0.078100,0.568945,633
2,123,0.485152,-0.007355,0.812500,0.558144,0.327224,0.088577,0.577342,624
3,2024,0.499988,0.013408,0.815032,0.564496,0.324421,0.071883,0.577784,632
4,99,0.487221,-0.000190,0.802694,0.556829,0.321200,0.076587,0.576237,631
5,11,0.482053,-0.008575,0.808791,0.559983,0.326309,0.080329,0.577416,637
6,77,0.478034,-0.010785,0.808634,0.561083,0.326009,0.084592,0.578595,637
7,314,0.494615,0.005167,0.809091,0.557512,0.329430,0.074013,0.570050,638
8,2718,0.487324,-0.000369,0.811146,0.562074,0.327615,0.078905,0.577048,637



Final test metrics variability across random seeds:


,cv_test_mean,cv_test_std,npmi_test_mean,npmi_test_std,topic_diversity_mean,topic_diversity_std,intra_topic_similarity_mean,intra_topic_similarity_std,inter_topic_similarity_mean,inter_topic_similarity_std,test_silhouette_mean,test_silhouette_std,test_outlier_ratio_mean,test_outlier_ratio_std,test_topic_count_mean,test_topic_count_std
0,0.488876,0.006797,-0.001116,0.007448,0.809396,0.003413,0.559386,0.004406,0.326377,0.002435,0.078814,0.005168,0.57627,0.00431,632.777778,5.093569



Thesis-ready reporting table (mean ± std across seeds):


,Model,Seeds (n),Coherence C_v (test),Coherence NPMI (test),Topic Diversity (test),Intra-topic Similarity (test),Inter-topic Similarity (test),"Silhouette (test, cosine)",Outlier Ratio (test),Topic Count (test)
0,BERTopic (best CV config),9,0.4889 ± 0.0068,-0.0011 ± 0.0074,0.8094 ± 0.0034,0.5594 ± 0.0044,0.3264 ± 0.0024,0.0788 ± 0.0052,0.5763 ± 0.0043,632.78 ± 5.09


Runs cluster diagnostics and visual sanity-check plots.

In [12]:
import plotly.express as px

# Diagnostics use the currently available final test assignment (`test_topics`) and embeddings (`test_embeddings`).
if 'test_topics' not in globals() or 'test_embeddings' not in globals():
    raise RuntimeError('Please run Cell 24 first so `test_topics` and `test_embeddings` are available.')

labels = np.asarray(test_topics)
emb = np.asarray(test_embeddings)

if labels.shape[0] != emb.shape[0]:
    raise ValueError(f'Mismatch: labels={labels.shape[0]} vs embeddings={emb.shape[0]}')

is_outlier = labels == -1
outlier_ratio_now = float(np.mean(is_outlier)) if len(labels) > 0 else np.nan

valid_labels = labels[~is_outlier]
if len(valid_labels) > 0:
    cluster_size_series = pd.Series(valid_labels).value_counts().sort_values(ascending=False)
else:
    cluster_size_series = pd.Series(dtype='int64')

n_valid_clusters_now = int(cluster_size_series.shape[0])
singleton_topics_now = set(cluster_size_series[cluster_size_series == 1].index.tolist())
n_singletons_now = int(len(singleton_topics_now))

print('Final-clustering diagnostics (current test assignment):')
print(f'  Documents: {len(labels)}')
print(f'  Outlier ratio (-1): {outlier_ratio_now:.4f}')
print(f'  Valid clusters (excluding -1): {n_valid_clusters_now}')
print(f'  Singleton clusters: {n_singletons_now}')

if n_valid_clusters_now == 0:
    print('⚠️ No valid clusters found (all points are outliers).')
elif n_valid_clusters_now == 1:
    print('⚠️ Only one valid cluster found. Silhouette is undefined by design.')
if n_singletons_now > 0:
    print('⚠️ Singleton clusters present. Silhouette excludes them in our robust function.')

# 1) Cluster size distribution (excluding outliers)
if n_valid_clusters_now > 0:
    cluster_size_df = (
        cluster_size_series
        .rename_axis('Topic')
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
    )
    cluster_size_df['Topic'] = cluster_size_df['Topic'].astype(int)

    fig_sizes = px.bar(
        cluster_size_df,
        x='Topic',
        y='Count',
        title='Final Test Cluster Sizes (excluding outliers)',
        labels={'Topic': 'Topic ID', 'Count': 'Number of Documents'}
    )
    fig_sizes.show()
else:
    print('Cluster-size plot skipped (no valid clusters).')

# 2) 2D embedding view colored by diagnostic point type
# Recompute a 2D projection from test embeddings for visual inspection.
umap_viz = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
emb_2d = umap_viz.fit_transform(emb)

point_type = np.where(
    labels == -1,
    'Outlier (-1)',
    np.where(np.isin(labels, list(singleton_topics_now)), 'Singleton cluster', 'Valid cluster')
)

viz_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'topic': labels.astype(int),
    'point_type': point_type
})

fig_diag = px.scatter(
    viz_df,
    x='x',
    y='y',
    color='point_type',
    hover_data=['topic'],
    title='Final Test Embedding Diagnostics (Outliers / Singleton / Valid)'
)
fig_diag.update_traces(marker=dict(size=6, opacity=0.7))
fig_diag.show()

# 3) Seed-level context from the multi-seed run
if 'seed_results' in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty:
    seed_plot_df = seed_results.copy()

    fig_seed_outliers = px.bar(
        seed_plot_df,
        x='seed',
        y='test_outlier_ratio',
        title='Outlier Ratio by Seed',
        labels={'seed': 'Seed', 'test_outlier_ratio': 'Outlier Ratio'}
    )
    fig_seed_outliers.show()

    fig_seed_topics = px.bar(
        seed_plot_df,
        x='seed',
        y='test_topic_count',
        title='Valid Topic Count by Seed',
        labels={'seed': 'Seed', 'test_topic_count': 'Topic Count (excluding -1)'}
    )
    fig_seed_topics.show()
else:
    print('Seed-level plots skipped (run Cell 24 first to populate `seed_results`).')

Final-clustering diagnostics (current test assignment):
  Documents: 13576
  Outlier ratio (-1): 0.5770
  Valid clusters (excluding -1): 496
  Singleton clusters: 94
⚠️ Singleton clusters present. Silhouette excludes them in our robust function.


Cluster diagnostics and visual sanity checks (outliers, valid clusters, singleton clusters)

## Scientific Tables, Visualizations, and Final Reporting

In [13]:
import plotly.express as px
import plotly.graph_objects as go

if 'df' not in globals():
    raise RuntimeError('Please run the data loading/preparation cells first so `df`, `train_df`, `val_df`, and `test_df` exist.')

if 'summary_stats' not in globals() or 'seed_results' not in globals():
    raise RuntimeError('Please run the multi-seed final evaluation cell first so `summary_stats` and `seed_results` exist.')

if 'tuning_results' not in globals() or 'cv_results' not in globals():
    raise RuntimeError('Please run the hyperparameter tuning cell first so `tuning_results` and `cv_results` exist.')

# ---------- Table 1: Corpus + Split statistics ----------
split_table = pd.DataFrame([
    {
        'Split': 'Train',
        'Documents': len(train_df),
        'Share (%)': 100 * len(train_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(train_df['date']).min(),
        'End': pd.to_datetime(train_df['date']).max(),
        'Unique Days': train_df['date'].dt.floor('D').nunique()
    },
    {
        'Split': 'Validation',
        'Documents': len(val_df),
        'Share (%)': 100 * len(val_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(val_df['date']).min(),
        'End': pd.to_datetime(val_df['date']).max(),
        'Unique Days': val_df['date'].dt.floor('D').nunique()
    },
    {
        'Split': 'Test',
        'Documents': len(test_df),
        'Share (%)': 100 * len(test_df) / len(df) if len(df) else np.nan,
        'Start': pd.to_datetime(test_df['date']).min(),
        'End': pd.to_datetime(test_df['date']).max(),
        'Unique Days': test_df['date'].dt.floor('D').nunique()
    }
])

corpus_table = pd.DataFrame([
    {
        'Total Documents': len(df),
        'Unique Headlines': int(df['headline'].nunique()) if 'headline' in df.columns else np.nan,
        'Date Start': pd.to_datetime(df['date']).min(),
        'Date End': pd.to_datetime(df['date']).max(),
        'Total Unique Days': df['date'].dt.floor('D').nunique()
    }
])

print('Table A — Corpus Overview')
display(corpus_table)
print('Table B — Chronological Split Design')
display(split_table)

# ---------- Table 2: Tuning leaderboard (scientific report style) ----------
leaderboard_cols = [
    'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples',
    'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity',
    'inter_topic_similarity', 'n_topics', 'fit_seconds', 'composite_score'
]

available_cols = [col for col in leaderboard_cols if col in tuning_results.columns]
leaderboard = tuning_results[available_cols].head(10).copy()

for col in ['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'composite_score', 'fit_seconds']:
    if col in leaderboard.columns:
        leaderboard[col] = pd.to_numeric(leaderboard[col], errors='coerce').round(4)

print('Table C — Top-10 Hyperparameter Configurations (rolling CV)')
display(leaderboard)

# ---------- Table 3: Final test performance (mean ± std across seeds) ----------
final_metrics = pd.DataFrame([
    {
        'Metric': 'Coherence C_v',
        'Mean': summary_stats.at[0, 'cv_test_mean'],
        'Std': summary_stats.at[0, 'cv_test_std']
    },
    {
        'Metric': 'Coherence NPMI',
        'Mean': summary_stats.at[0, 'npmi_test_mean'],
        'Std': summary_stats.at[0, 'npmi_test_std']
    },
    {
        'Metric': 'Topic Diversity',
        'Mean': summary_stats.at[0, 'topic_diversity_mean'],
        'Std': summary_stats.at[0, 'topic_diversity_std']
    },
    {
        'Metric': 'Intra-topic Similarity',
        'Mean': summary_stats.at[0, 'intra_topic_similarity_mean'],
        'Std': summary_stats.at[0, 'intra_topic_similarity_std']
    },
    {
        'Metric': 'Inter-topic Similarity',
        'Mean': summary_stats.at[0, 'inter_topic_similarity_mean'],
        'Std': summary_stats.at[0, 'inter_topic_similarity_std']
    },
    {
        'Metric': 'Silhouette (cosine)',
        'Mean': summary_stats.at[0, 'test_silhouette_mean'],
        'Std': summary_stats.at[0, 'test_silhouette_std']
    },
    {
        'Metric': 'Outlier Ratio',
        'Mean': summary_stats.at[0, 'test_outlier_ratio_mean'],
        'Std': summary_stats.at[0, 'test_outlier_ratio_std']
    },
    {
        'Metric': 'Topic Count',
        'Mean': summary_stats.at[0, 'test_topic_count_mean'],
        'Std': summary_stats.at[0, 'test_topic_count_std']
    }
])

final_metrics['Mean'] = pd.to_numeric(final_metrics['Mean'], errors='coerce').round(4)
final_metrics['Std'] = pd.to_numeric(final_metrics['Std'], errors='coerce').round(4)
final_metrics['Mean ± Std'] = final_metrics.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}" if pd.notna(row['Mean']) and pd.notna(row['Std']) else 'NA',
    axis=1
)

print('Table D — Final Test Metrics Across Seeds')
display(final_metrics[['Metric', 'Mean ± Std', 'Mean', 'Std']])

Table A — Corpus Overview


,Total Documents,Unique Headlines,Date Start,Date End,Total Unique Days
0,90048,90048,2014-02-17 05:11:00+00:00,2026-04-11 17:07:00+00:00,3302


Table B — Chronological Split Design


,Split,Documents,Share (%),Start,End,Unique Days
0,Train,62969,69.928260,2014-02-17 05:11:00+00:00,2025-06-16 22:56:30+00:00,3003
1,Validation,13503,14.995336,2025-06-17 02:28:37+00:00,2025-11-18 23:36:34+00:00,155
2,Test,13576,15.076404,2025-11-19 00:09:00+00:00,2026-04-11 17:07:00+00:00,144


Table C — Top-10 Hyperparameter Configurations (rolling CV)


,n_neighbors,n_components,min_cluster_size,min_samples,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,composite_score
0,30,10,20,3,0.5752,0.0716,0.8442,0.4525,0.4179,171.000000,72.4726,0.7747
1,30,10,15,3,0.5612,0.0639,0.8473,0.4662,0.3976,222.666667,75.7710,0.6637
2,50,10,10,3,0.5645,0.0700,0.8247,0.5049,0.3562,286.000000,84.2491,0.6514
3,50,5,20,1,0.5571,0.0546,0.8556,0.4565,0.4162,172.666667,82.3431,0.6363
4,50,5,20,3,0.5588,0.0552,0.8544,0.4567,0.4068,156.666667,80.7210,0.6237
5,50,10,15,3,0.5515,0.0550,0.8490,0.4938,0.3844,196.333333,82.6302,0.6136
6,50,5,15,3,0.5553,0.0563,0.8500,0.4782,0.3855,200.666667,86.3638,0.6054
7,50,10,20,3,0.5561,0.0497,0.8501,0.4609,0.4105,153.666667,81.4533,0.5877
8,50,10,20,1,0.5533,0.0547,0.8408,0.4668,0.4013,172.000000,81.8030,0.5684
9,50,10,15,1,0.5549,0.0460,0.8449,0.4878,0.3791,226.000000,83.0403,0.5447


Table D — Final Test Metrics Across Seeds


,Metric,Mean ± Std,Mean,Std
0,Coherence C_v,0.4889 ± 0.0068,0.4889,0.0068
1,Coherence NPMI,-0.0011 ± 0.0074,-0.0011,0.0074
2,Topic Diversity,0.8094 ± 0.0034,0.8094,0.0034
3,Intra-topic Similarity,0.5594 ± 0.0044,0.5594,0.0044
4,Inter-topic Similarity,0.3264 ± 0.0024,0.3264,0.0024
5,Silhouette (cosine),0.0788 ± 0.0052,0.0788,0.0052
6,Outlier Ratio,0.5763 ± 0.0043,0.5763,0.0043
7,Topic Count,632.7778 ± 5.0936,632.7778,5.0936


Generates scientific figures for trends, topic prevalence, tuning trade-offs, and stability.

In [15]:
# ---------- Figure 1: Document intensity over time (all splits) ----------
plot_df = df[['date']].copy()
plot_df['date'] = pd.to_datetime(plot_df['date'])
plot_df['day'] = plot_df['date'].dt.floor('D')

daily_counts = plot_df.groupby('day').size().reset_index(name='documents')

split_dates = {
    'train_end': pd.to_datetime(train_df['date']).max(),
    'val_start': pd.to_datetime(val_df['date']).min(),
    'val_end': pd.to_datetime(val_df['date']).max(),
    'test_start': pd.to_datetime(test_df['date']).min()
}

fig_daily = px.line(
    daily_counts,
    x='day',
    y='documents',
    title='Daily Document Volume Over Time',
    labels={'day': 'Date', 'documents': 'Number of Documents'}
)


def add_date_marker(fig, date_value, label, color):
    if pd.isna(date_value):
        return
    marker_x = pd.to_datetime(date_value).to_pydatetime()
    fig.add_shape(
        type='line',
        x0=marker_x,
        x1=marker_x,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(color=color, dash='dash', width=1.5)
    )
    fig.add_annotation(
        x=marker_x,
        y=1,
        xref='x',
        yref='paper',
        yshift=10,
        text=label,
        showarrow=False,
        font=dict(color=color)
    )


add_date_marker(fig_daily, split_dates['train_end'], 'Train End', 'orange')
add_date_marker(fig_daily, split_dates['val_end'], 'Validation End', 'red')
fig_daily.update_layout(template='plotly_white')
fig_daily.show()

# ---------- Figure 2: Topic prevalence in final test assignment ----------
if 'test_topics' in globals():
    topic_counts = pd.Series(test_topics).value_counts(dropna=False).rename_axis('topic').reset_index(name='count')
    topic_counts['topic'] = topic_counts['topic'].astype(int)
    topic_counts = topic_counts.sort_values('count', ascending=False)

    fig_topic_prev = px.bar(
        topic_counts.head(20),
        x='topic',
        y='count',
        color='topic',
        title='Top 20 Topics by Test-Set Frequency (includes outlier topic -1)',
        labels={'topic': 'Topic ID', 'count': 'Document Count'}
    )
    fig_topic_prev.update_layout(showlegend=False, template='plotly_white')
    fig_topic_prev.show()
else:
    print('Topic prevalence figure skipped (`test_topics` not available).')

# ---------- Figure 3: Hyperparameter trade-off map ----------
if {'cv_val', 'topic_diversity', 'composite_score', 'n_neighbors', 'min_cluster_size'}.issubset(tuning_results.columns):
    fig_tradeoff = px.scatter(
        tuning_results,
        x='cv_val',
        y='topic_diversity',
        color='composite_score',
        size='n_topics' if 'n_topics' in tuning_results.columns else None,
        hover_data=['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'fit_seconds'],
        title='Tuning Trade-off: Coherence vs Diversity (colored by composite score)',
        labels={'cv_val': 'Validation Coherence (C_v)', 'topic_diversity': 'Topic Diversity', 'composite_score': 'Composite Score'}
    )
    fig_tradeoff.update_layout(template='plotly_white')
    fig_tradeoff.show()

# ---------- Figure 4: Seed stability profile ----------
if not seed_results.empty:
    long_seed = seed_results.melt(
        id_vars=['seed'],
        value_vars=['cv_test', 'npmi_test', 'topic_diversity_test', 'test_silhouette', 'test_outlier_ratio', 'test_topic_count'],
        var_name='metric',
        value_name='value'
    )

    fig_seed_box = px.box(
        long_seed,
        x='metric',
        y='value',
        points='all',
        color='metric',
        title='Final Model Stability Across Random Seeds',
        labels={'metric': 'Metric', 'value': 'Observed Value'}
    )
    fig_seed_box.update_layout(showlegend=False, template='plotly_white')
    fig_seed_box.show()

# ---------- Figure 5: Fold-level metric heatmap ----------
if {'fold', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity'}.issubset(cv_results.columns):
    fold_summary = cv_results.groupby('fold', as_index=False)[
        ['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity']
    ].mean()

    z_values = fold_summary[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity']].to_numpy()

    fig_heat = go.Figure(data=go.Heatmap(
        z=z_values,
        x=['C_v', 'NPMI', 'Diversity', 'Intra Sim', 'Inter Sim'],
        y=[f"Fold {int(f)}" for f in fold_summary['fold']],
        colorscale='Viridis',
        colorbar=dict(title='Mean Value')
    ))
    fig_heat.update_layout(
        title='Fold-wise Average Metrics During Hyperparameter Tuning',
        xaxis_title='Metric',
        yaxis_title='Time-Series CV Fold',
        template='plotly_white'
    )
    fig_heat.show()

print('Scientific visualizations generated.')

Scientific visualizations generated.
